# Description

W tym notatniku przeprowadzane są wszelkie eksperymenty, zarówno dla autoenkodera wariacyjnego i nie wariacyjnego, dla wszystkich członów funkcji straty, w wersji z douczaniem i bez (łącznie 12 eksperymentów)

# Imports

In [12]:
IS_NEW_APPROACH = True

In [13]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import os
import tqdm
import wandb
import json

sys.path.append('../')  # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.KlejdaGraphAutoencoder import KlejdaGraphAutoencoder
from src.models.KlejdaGAE.KlejdaVariationalGraphAutoencoder import KlejdaVariationalGraphAutoencoder
from src.models.NewGAE.GraphAutoencoder import GraphAutoencoder
from src.models.NewGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from pyprojroot import here

current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
from FramsticksLib import FramsticksLib
from deap import tools, algorithms
import yaml
from src.deap.deap_setup import prepare_native_toolbox, prepare_cmaes_toolbox
from src.deap.constraints import is_feasible_fitness_criteria
from src.deap.save_and_load_results import save_genotypes_json
from src.deap.custom_ea_algorithms import run_cma_es_with_validation
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from src.deap.AutoencoderEvaluator import AutoencoderEvaluator
import numpy as np
import frams
from copy import deepcopy

import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Pre-processing

In [14]:
project_dir = here()
# Przygotowanie checkpointów nauczonych autoenkoderów
checkpoints_dir = project_dir / 'notebooks' / 'checkpoints' / 'final_checkpoints'
if IS_NEW_APPROACH:
	checkpoints_dir = checkpoints_dir / 'new'
else:
	checkpoints_dir = checkpoints_dir / 'klejda'

checkpoint_gae = torch.load(checkpoints_dir / 'gae.ckpt')
checkpoint_vgae = torch.load(checkpoints_dir / 'vgae.ckpt')
wandb.login()

True

In [15]:
# Przygotowanie konfiguracji dla gae
configs_dir = project_dir / 'configs'
if IS_NEW_APPROACH:
	config_gae_path = configs_dir / 'gae_config_large.yaml'
else:
	config_gae_path = configs_dir / 'klejda_gae_config.yaml'
with open(config_gae_path) as f:
	config_gae = yaml.safe_load(f)
	config_vgae = config_gae

In [48]:
# # Przygotowanie środowiska Framsticks oraz DEAP
# with open("../configs/final_evolution_config.yaml", 'r') as f:
# 	evolution_config = yaml.safe_load(f)
# frams.init(
#     evolution_config['frams_path']
# )
# frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])
#
# toolbox = prepare_native_toolbox(frams_lib, evolution_config)
# # TODO: TO dodać przed uruchomieniem eksperymentu
# # toolbox.register("mutate", autoencoder_mutate, gae)
# pop = toolbox.population(n=evolution_config['pop_size'])
# hof = tools.HallOfFame(evolution_config['hof_size'])
#
# stats = tools.Statistics(lambda ind: ind.fitness.values)
# filter_feasible = lambda func, criteria: func(list(filter(is_feasible_fitness_criteria, criteria)))
# stats.register("min", lambda fit: filter_feasible(np.min, fit))
# stats.register("avg", lambda fit: filter_feasible(np.mean, fit))
# stats.register("max", lambda fit: filter_feasible(np.max, fit))
#
# postProcessingGaeResult = FramsticksPostProcessor()

In [16]:
# Wersja CMA-ES
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)
frams.init(
	evolution_config['frams_path']
)
frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])
toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)


hof = tools.HallOfFame(evolution_config['hof_size'])
stats = tools.Statistics(lambda ind: ind.fitness.values)
def safe_filter_feasible(func, criteria):
    feasible_fits = list(filter(is_feasible_fitness_criteria, criteria))
    if len(feasible_fits) == 0:
        return np.nan
    return round(func(feasible_fits),3)
stats.register("min", lambda fit: safe_filter_feasible(np.min, fit))
stats.register("avg", lambda fit: safe_filter_feasible(np.mean, fit))
stats.register("max", lambda fit: safe_filter_feasible(np.max, fit))

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Available objects: ['CheckpointEvent', 'Collision', 'CrCollision', 'Creature', 'CreatureSettings', 'CreatureSignals', 'CreatureSnapshot', 'Dictionary', 'ExpProperties', 'ExpState', 'ExtValue', 'File', 'FunctionReference', 'GenMan', 'GenManStats', 'GenePool', 'GenePools', 'Geno', 'GenoConverters', 'Genotype', 'Interface', 'Joint', 'Loader', 'Math', 'MechJoint', 'MechPart', 'MessageCatcher', 'Model', 'ModelGeometry', 'ModelSymmetry', 'Neuro', 'NeuroClass', 'NeuroClassLibrary', 'NeuroDef', 'NeuroSignals', 'NeuronsSimEnabled', 'ODE', 'Orient', 'Part', 'Pop

# Experiments

## GAE

In [7]:
if IS_NEW_APPROACH:
	gae_non_cyclic = GraphAutoencoder(config=config_gae, frams_module=frams).double()
else:
	gae_non_cyclic = KlejdaGraphAutoencoder(config=config_gae, frams_module=frams).double()
gae_non_cyclic.load_state_dict(checkpoint_gae['state_dict'])
gae_non_cyclic.eval()
evaluator = AutoencoderEvaluator(gae_non_cyclic, frams_lib,evolution_config['opt_criteria'], evolution_config)
toolbox.register("evaluate", evaluator)

### Trained once

In [8]:
pop, log, reconstruction_ratio, _ = run_cma_es_with_validation(
	 toolbox,
	ngen = evolution_config['generations'],
	stats=stats,
	halloffame=hof,
	verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x3 and 15x64)

### Continual training

In [39]:
with open("../configs/klejda_gae_config.yaml") as f:
    config = yaml.safe_load(f)


for i in range(3):
	# Krok 1: Przeprowadzenie ewolucji
	toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)
	toolbox.register("evaluate", evaluator)

	pop, log, reconstruction_above_threshold, created_individuals = run_cma_es_with_validation(
		 toolbox,
		ngen = evolution_config['generations'],
		stats=stats,
		halloffame=hof,
		verbose=True,
		validity_threshold=0.5
	)

	# Krok 2: Douczenie autoenkodera
	# Zakończono ewolucję z powodu spadku jakości rekonstrukcji
	if not reconstruction_above_threshold:
		pass

	genotypes = []
	for ind in created_individuals:
		genotypes.append({'genotype': ind.genotype, 'fitness': ind.fitness.values})
	dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])

	dataset_size = len(dataset)
	train_size = int(0.8 * dataset_size)
	val_size = dataset_size - train_size
	train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

	wandb_logger = WandbLogger(project="Framsticks-GAE", name="GAE-Baseline-Test", save_dir = config["save_dir"])


	train_dataloader = DataLoader(
	    train_dataset,
	    batch_size=256,
	    shuffle=True,
	    num_workers=4,
	    persistent_workers=True
	)

	val_dataloader = DataLoader(
	    val_dataset,
	    batch_size=256,
	    shuffle=False,
	    num_workers=4,
	    persistent_workers=True
	)
	trainer = pl.Trainer(
	    max_epochs=160,
	    logger=wandb_logger,
	    log_every_n_steps=5,
	    accelerator="auto",
	    devices=1
	)
	gae_non_cyclic.train()
	trainer.fit(gae_non_cyclic, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
	gae_non_cyclic.eval()
	wandb.finish()

gen	nevals	valid_recon_ratio	F_ZERO_LEN	F_SUBGROUPS	F_LONG_PARTS	valid_frams_ratio	min   	avg  	max  
0  	140   	1                	0.01      	0.07       	0.64        	0.14             	-0.003	0.195	0.564
1  	140   	0.99             	0         	0.06       	0.71        	0.16             	0.028 	0.268	1.756
2  	140   	1                	0.01      	0.08       	0.71        	0.11             	0.05  	0.192	0.496
3  	140   	0.99             	0         	0.1        	0.71        	0.21             	0.043 	0.188	0.553
4  	140   	1                	0         	0.1        	0.65        	0.2              	0.016 	0.15 	0.4  
5  	140   	1                	0         	0.08       	0.69        	0.22             	0.006 	0.256	0.863
6  	140   	0.99             	0         	0.09       	0.71        	0.19             	0.038 	0.218	1.293
7  	140   	1                	0         	0.06       	0.7         	0.24             	0.002 	0.159	0.403
8  	140   	0.99             	0         	0.09       	0.69        	0.19             

KeyboardInterrupt: 

## VGAE

### Trained once

In [17]:
if IS_NEW_APPROACH:
	vgae_non_cyclic = VariationalGraphAutoencoder(config=config_gae, frams_module=frams).double()
else:
	vgae_non_cyclic = KlejdaVariationalGraphAutoencoder(config=config_gae, frams_module=frams).double()
vgae_non_cyclic.load_state_dict(checkpoint_vgae['state_dict'])
vgae_non_cyclic.eval()
evaluator = AutoencoderEvaluator(vgae_non_cyclic, frams_lib,evolution_config['opt_criteria'], evolution_config)
toolbox.register("evaluate", evaluator)

In [18]:
pop, log, reconstruction_ratio, _ = run_cma_es_with_validation(
	 toolbox,
	ngen = evolution_config['generations'],
	stats=stats,
	halloffame=hof,
	verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	valid_recon_ratio	F_ZERO_LEN	F_SUBGROUPS	F_LONG_PARTS	valid_frams_ratio	min  	avg  	max  
0  	140   	1                	0         	0.71       	0.04        	1                	0.001	0.107	0.273
1  	140   	1                	0         	0.82       	0.04        	1                	-0.008	0.098	0.282
2  	140   	1                	0         	0.81       	0.05        	0.99             	0.015 	0.117	0.337
3  	140   	1                	0         	0.85       	0.06        	1                	-0.004	0.138	0.542
4  	140   	1                	0         	0.83       	0.03        	1                	0.025 	0.162	0.452
5  	140   	1                	0         	0.82       	0.09        	0.99             	0.027 	0.2  	0.581
6  	140   	1                	0         	0.77       	0.14        	1                	0.013 	0.217	0.459
7  	140   	1                	0         	0.7        	0.24        	0.99             	0.078 	0.248	0.671
8  	140   	1                	0         	0.73       	0.33        	1                	-

### Continual training

In [ ]:
with open("../configs/klejda_vgae_config.yaml") as f:
    config = yaml.safe_load(f)

loaded_genotypes = []
with open("../results/sampled_best_individuals_merged.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        loaded_genotypes.append(obj)

genotypes = deepcopy(loaded_genotypes)
for i in range(3):
	# Krok 1: Przeprowadzenie ewolucji
	toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)
	toolbox.register("evaluate", evaluator)

	pop, log, reconstruction_above_threshold, created_individuals = run_cma_es_with_validation(
		 toolbox,
		ngen = evolution_config['generations'],
		stats=stats,
		halloffame=hof,
		verbose=True,
		validity_threshold=0.5
	)

	# Krok 2: Douczenie autoenkodera
	# Zakończono ewolucję z powodu spadku jakości rekonstrukcji
	if not reconstruction_above_threshold:
		pass


	for ind in created_individuals:
		genotypes.append({'genotype': ind.genotype, 'fitness': ind.fitness.values})
	dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])

	dataset_size = len(dataset)
	train_size = int(0.8 * dataset_size)
	val_size = dataset_size - train_size
	train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

	wandb_logger = WandbLogger(project="Framsticks-VGAE", name="VGAE-Baseline-Test", save_dir = config["save_dir"])


	train_dataloader = DataLoader(
	    train_dataset,
	    batch_size=256,
	    shuffle=True,
	    num_workers=4,
	    persistent_workers=True
	)

	val_dataloader = DataLoader(
	    val_dataset,
	    batch_size=256,
	    shuffle=False,
	    num_workers=4,
	    persistent_workers=True
	)
	trainer = pl.Trainer(
	    max_epochs=160,
	    logger=wandb_logger,
	    log_every_n_steps=5,
	    accelerator="auto",
	    devices=1
	)
	vgae_non_cyclic.train()
	trainer.fit(vgae_non_cyclic, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
	vgae_non_cyclic.eval()
	wandb.finish()

gen	nevals	valid_recon_ratio	F_ZERO_LEN	F_SUBGROUPS	F_LONG_PARTS	valid_frams_ratio	min	avg  	max  
0  	140   	1                	0         	0.1        	0.74        	0.13             	-0 	0.147	0.478
1  	140   	1                	0         	0.14       	0.7         	0.2              	-0.01	0.11 	0.256
2  	140   	1                	0         	0.09       	0.69        	0.16             	-0.01	0.135	0.625
3  	140   	1                	0         	0.09       	0.64        	0.26             	-0.01	0.102	0.391
4  	140   	1                	0         	0.11       	0.61        	0.21             	-0.01	0.129	0.866
5  	140   	1                	0         	0.11       	0.54        	0.35             	-0.01	0.121	0.781
6  	140   	1                	0         	0.11       	0.59        	0.39             	-0.01	0.116	1.039
7  	140   	1                	0         	0.06       	0.59        	0.39             	-0.01	0.113	0.455
8  	140   	1                	0         	0.04       	0.64        	0.36             	-0.01	0.107	

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


99 	140   	1                	0         	0          	0.36        	0.79             	0.085 	0.679	1.332


wandb: setting up run c5ag6yqn
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260816_214645-c5ag6yqn
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run VGAE-Baseline-Test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-VGAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-VGAE/runs/c5ag6yqn
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │ 70.9 K │ train │     0 │
│ 1 │ fc_mu            │ Linear   │  6.5 K │ train │     0 │
│ 2 │ fc_logvar        │ Linear   │  6.5 K │ train │     0 │
│ 3 │ decoder_a        │ DecoderA │ 44.2 K │ train │     0 │
│ 4 │ decoder_x        │ DecoderX │  189 K │ train │     0 │
│ 5 │ criterion        │ MSELoss  │      0 │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


99 	140   	1                	0         	0          	1           	0.21             	0.23  	1.494	1.548


wandb: setting up run whdtmuis
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260816_220804-whdtmuis
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run VGAE-Baseline-Test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-VGAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-VGAE/runs/whdtmuis
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │ 70.9 K │ train │     0 │
│ 1 │ fc_mu            │ Linear   │  6.5 K │ train │     0 │
│ 2 │ fc_logvar        │ Linear   │  6.5 K │ train │     0 │
│ 3 │ decoder_a        │ DecoderA │ 44.2 K │ train │     0 │
│ 4 │ decoder_x        │ DecoderX │  189 K │ train │     0 │
│ 5 │ criterion        │ MSELoss  │      0 │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 317 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 317 K                                                                                                
Total estimated model params size (MB): 1.269                                                                      
Modules in train mode: 70                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

# Results